# grad-accumulate-on-leaf — worked example 3: Training loop: accumulate gradients across steps then zero

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-accumulate-on-leaf`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In a standard training loop, gradient accumulation across backward passes within a single step is INTENTIONAL (e.g., micro-batching). Gradient leakage ACROSS steps is a bug. The fix is `zero_grad` (setting `.grad = None`) at the boundary between steps. Setting to `None` rather than to zeros is preferred because it makes the next backward's first-touch path allocate fresh rather than adding to a zero tensor — a minor efficiency gain and semantically cleaner.

## Worked solution

**Step 1 — simulate two training steps with a single scalar parameter.** We track a list of recorded gradients to verify the per-step behavior.

**Step 2 — step 1: two micro-batches.** We call `accumulate_grad` twice to simulate two micro-batch contributions. After both calls, `param.grad` should be the sum of both contributions.

**Step 3 — record step 1 grad.** Save the total for comparison.

**Step 4 — zero_grad between steps.** Call `zero_grad` (set `.grad = None`). This prevents step-1's gradient from bleeding into step-2.

**Step 5 — step 2: different micro-batches.** Same accumulation pattern but with different gradient values. The result should be only the step-2 contributions, not step-1 + step-2.

**Step 6 — confirm no leakage.** The step-2 gradient should NOT contain any of the step-1 values.

In [ ]:
import torch as t

t.manual_seed(7)

class SimpleLeaf:
    def __init__(self):
        self.grad = None

def accumulate_grad(leaf, g):
    if leaf.grad is None:
        leaf.grad = g
    else:
        leaf.grad = leaf.grad + g

def zero_grad(params):
    for p in params:
        p.grad = None

param = SimpleLeaf()

# === Step 1: two micro-batch gradients ===
g_step1_batch1 = t.tensor([3.0, -1.0])
g_step1_batch2 = t.tensor([1.0,  2.0])

accumulate_grad(param, g_step1_batch1)
accumulate_grad(param, g_step1_batch2)
step1_grad = param.grad.clone()
print(f"Step 1 grad: {step1_grad}")  # [4.0, 1.0]
assert t.allclose(step1_grad, g_step1_batch1 + g_step1_batch2)

# === zero_grad between steps ===
zero_grad([param])
assert param.grad is None, "zero_grad must set .grad = None"
print("After zero_grad: param.grad is None")

# === Step 2: two different micro-batch gradients ===
g_step2_batch1 = t.tensor([0.5, 0.5])
g_step2_batch2 = t.tensor([0.5, 0.5])

accumulate_grad(param, g_step2_batch1)
accumulate_grad(param, g_step2_batch2)
step2_grad = param.grad
print(f"Step 2 grad: {step2_grad}")  # [1.0, 1.0]
assert t.allclose(step2_grad, g_step2_batch1 + g_step2_batch2)

# Confirm no leakage from step 1
assert not t.allclose(step2_grad, step1_grad + g_step2_batch1 + g_step2_batch2)
print("Confirmed: no gradient leakage across steps.")